# DCVC-RT-VCM — RANDOM QP + λ(QP) 1→64 — SINGLE MODEL

Notebook train **một model variable-rate duy nhất** theo cơ chế rate control của DCVC-RT và objective Base-layer của SVC:

- **DMCI:** pretrained, frozen; tạo I-frame reference.
- **DMC:** pretrained, fine-tune.
- **YOLOv5s teacher:** frozen; **student front 5 layer:** train cùng DMC.
- **Base QP:** lấy ngẫu nhiên đều `q ∈ {0,…,63}` ở mỗi iteration và đồng bộ giữa 2 GPU.
- **Ánh xạ:** `λ(q) = 64^(q/63)`, nên `q=(0,21,42,63) → λ=(1,4,16,64)`.
- **QP phân cấp:** offset theo GOP `[0,8,0,4,0,4,0,4]`.
- **Trọng số distortion:** lặp `(0.5,1.2,0.5,0.9)`.
- **Loss mỗi P-frame:** `BPP + λ(q) × w_t × Feature_MSE`; loss clip là trung bình 6 P-frame.
- Giữ cấu hình train cũ: Vimeo-90K, crop 256, Adam 1e-6, effective batch 4, 2×GPU, tối đa 40 epoch; dừng sớm khi validation loss hội tụ.
- **Theo dõi:** rank 0 log Loss/BPP/Feature MSE và ảnh đường cong từng epoch lên Comet.

Checkpoint cũ của bốn λ cố định không được resume vào experiment này.


In [ ]:
# 1) Kiểm tra môi trường Kaggle — hãy chọn Accelerator = GPU T4 x2 trước khi chạy
import os, sys, torch, platform

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} | VRAM={p.total_memory/1024**3:.1f} GB')

assert torch.cuda.is_available(), 'CUDA chưa bật.'
assert torch.cuda.device_count() >= 2, 'Notebook này được cấu hình cho 2 GPU. Hãy chọn GPU T4 x2.'

In [ ]:
# ============================================================
# CELL 2 — MAIN CONFIG
# Một model: random QP 0..63, λ(q)=64^(q/63)
# ============================================================

import json
import math
import os

CFG = {
    'vimeo_root': None,
    'image_ckpt': None,
    'video_ckpt': None,

    'lambda_min': 1.0,
    'lambda_max': 64.0,
    'epochs': 40,
    'early_stopping_patience': 5,
    'early_stopping_min_delta': 1e-4,

    'lr_video': 1e-6,
    'lr_yolo': 1e-6,
    'weight_decay': 0.0,
    'crop_size': 256,

    # 1 I-frame reference + 6 P-frames; dùng đủ 7 frame.
    'p_frames': 6,

    # Global batch = 2 sample/GPU × 2 GPU = 4; không cần gradient accumulation.
    'batch_per_gpu': 2,
    'grad_accum': 1,
    'num_workers': 2,

    'qp_min': 0,
    'qp_max': 63,

    'grad_clip': 1.0,
    'amp': True,
    'seed': 1234,
    'max_train_samples': 0,
    'max_val_samples': 100,

    'val_qps': '0,21,42,63',
    'eval_qps': '0,8,16,24,32,40,48,56,63',

    # Tách hoàn toàn khỏi output bốn lambda cũ.
    'output_root': '/kaggle/working/dcvc_rt_vcm_random_qp_lambda_1_64_7frame',
}

RUN_SMOKE_TEST = False
RUN_FULL_TRAIN = True
MAX_STEPS = 0
AUTO_RESUME = True

# Tạo Kaggle Secret tên COMET_API_KEY trước khi chạy.
USE_COMET = True
COMET_PROJECT_NAME = 'dcvc-rt-vcm'
if USE_COMET:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['COMET_API_KEY'] = UserSecretsClient().get_secret('COMET_API_KEY')
        os.environ['COMET_PROJECT_NAME'] = COMET_PROJECT_NAME
    except Exception as exc:
        raise RuntimeError(
            'Chưa có Kaggle Secret COMET_API_KEY. Vào Add-ons > Secrets để thêm, '
            'hoặc đặt USE_COMET=False.'
        ) from exc

QP_OFFSETS = [0, 8, 0, 4, 0, 4, 0, 4]
DISTORTION_WEIGHTS = [0.5, 1.2, 0.5, 0.9]

def mapped_lambda(q):
    return CFG['lambda_min'] * (
        CFG['lambda_max'] / CFG['lambda_min']
    ) ** ((q - CFG['qp_min']) / (CFG['qp_max'] - CFG['qp_min']))

assert CFG['qp_min'] == 0 and CFG['qp_max'] == 63
assert CFG['lambda_min'] == 1.0 and CFG['lambda_max'] == 64.0
assert all(math.isclose(mapped_lambda(q), expected) for q, expected in (
    (0, 1.0), (21, 4.0), (42, 16.0), (63, 64.0)
))
assert MAX_STEPS == 0

print(json.dumps(CFG, indent=2))
print('QP offsets:', QP_OFFSETS)
print('Distortion weights:', DISTORTION_WEIGHTS)
print('Landmarks:', {q: mapped_lambda(q) for q in (0, 21, 42, 63)})


In [ ]:
# 3) Auto-discover Vimeo-90K septuplet 7 frame + checkpoints
from pathlib import Path

INPUT = Path('/kaggle/input')
print('Attached Kaggle datasets:')
for p in sorted(INPUT.iterdir()):
    print(' -', p)


def find_vimeo_root():
    roots = []
    for train_list in INPUT.rglob('sep_trainlist.txt'):
        root = train_list.parent
        if (root / 'sep_testlist.txt').is_file() and (root / 'sequences').is_dir():
            roots.append(root)
    if not roots:
        return None
    return min(roots, key=lambda p: ('vimeo-90k' not in str(p).lower(), len(str(p))))


def ckpt_candidates():
    exts = ('.pth', '.pt', '.ckpt', '.tar')
    return [
        p for p in INPUT.rglob('*')
        if p.is_file() and (p.name.endswith('.pth.tar') or p.suffix.lower() in exts)
    ]


def score_ckpt(p, kind):
    n = p.name.lower()
    s = 0
    if 'cvpr2025' in n: s += 20
    if 'dcvc' in str(p).lower(): s += 5
    if kind in n: s += 20
    if kind == 'image' and ('intra' in n or '_i' in n): s += 5
    if kind == 'video' and ('video' in n or '_p' in n): s += 5
    if kind == 'image' and 'video' in n: s -= 20
    if kind == 'video' and 'image' in n: s -= 20
    return s


vimeo_auto = find_vimeo_root()
cks = ckpt_candidates()
image_auto = max(cks, key=lambda p: score_ckpt(p, 'image')) if cks else None
video_auto = max(cks, key=lambda p: score_ckpt(p, 'video')) if cks else None

VIMEO_ROOT = Path(CFG['vimeo_root']) if CFG['vimeo_root'] else vimeo_auto
IMAGE_CKPT = Path(CFG['image_ckpt']) if CFG['image_ckpt'] else image_auto
VIDEO_CKPT = Path(CFG['video_ckpt']) if CFG['video_ckpt'] else video_auto

print('\nResolved paths:')
print('VIMEO_ROOT =', VIMEO_ROOT)
print('IMAGE_CKPT =', IMAGE_CKPT)
print('VIDEO_CKPT =', VIDEO_CKPT)

assert VIMEO_ROOT is not None, (
    'Không tìm thấy Vimeo-90K septuplet có sep_trainlist.txt, '
    'sep_testlist.txt và sequences/. Không dùng vimeo_triplet 3 frame.'
)
assert (VIMEO_ROOT / 'sep_trainlist.txt').is_file()
assert (VIMEO_ROOT / 'sep_testlist.txt').is_file()
assert (VIMEO_ROOT / 'sequences').is_dir()
assert IMAGE_CKPT is not None and IMAGE_CKPT.is_file(), 'Không tìm được cvpr2025_image.pth.tar.'
assert VIDEO_CKPT is not None and VIDEO_CKPT.is_file(), 'Không tìm được cvpr2025_video.pth.tar.'

train_lines = [
    x.strip()
    for x in (VIMEO_ROOT / 'sep_trainlist.txt').read_text().splitlines()
    if x.strip()
]
val_lines = [
    x.strip()
    for x in (VIMEO_ROOT / 'sep_testlist.txt').read_text().splitlines()
    if x.strip()
]
assert train_lines and val_lines, 'Train/test list đang rỗng.'

example = VIMEO_ROOT / 'sequences' / train_lines[0]
missing_frames = [
    str(example / f'im{i}.png')
    for i in range(1, 8)
    if not (example / f'im{i}.png').is_file()
]
assert not missing_frames, (
    'Dataset không đủ 7 frame. Thiếu:\n' + '\n'.join(missing_frames)
)

print('\n' + '=' * 80)
print('ACTUAL DATASET COUNTS FROM ATTACHED FILES')
print('=' * 80)
print('TRAIN sequences =', len(train_lines))
print('TEST/VAL sequences =', len(val_lines))

world = 2
per_rank_sampler_samples = len(train_lines) // world
steps_per_rank = per_rank_sampler_samples // CFG['batch_per_gpu']
effective_sequences = steps_per_rank * CFG['batch_per_gpu'] * world

print('DDP world size                      =', world)
print('batch_per_gpu                       =', CFG['batch_per_gpu'])
print('expected train steps / GPU / epoch  =', steps_per_rank)
print('effective sequences / full epoch    =', effective_sequences)
print('NOTE: train uses all 7 frames: im1 is I-frame, im2..im7 are 6 P-frames.')
print('\nExample sequence:', example)
for i in range(1, 8):
    print(f'im{i}.png:', (example / f'im{i}.png').is_file())


In [ ]:
# 4) Clone đúng source DCVC-RT + YOLOv5 v7.0 và cài dependency nhẹ
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
DCVC_REPO = WORK/'DCVC'
DCVC_RT = DCVC_REPO/'DCVC-family'/'DCVC-RT'
YOLO_REPO = WORK/'yolov5'
YOLO_WEIGHTS = WORK/'yolov5s.pt'

if not DCVC_REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/microsoft/DCVC.git',str(DCVC_REPO)], check=True)
else:
    print('DCVC already exists:', DCVC_REPO)

if not YOLO_REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','v7.0','https://github.com/ultralytics/yolov5.git',str(YOLO_REPO)], check=True)
else:
    print('YOLOv5 already exists:', YOLO_REPO)

# Không thay torch/torchvision của Kaggle.
subprocess.run([sys.executable,'-m','pip','install','-q','pybind11','bd-metric','thop','pyyaml','comet_ml'], check=True)

if not YOLO_WEIGHTS.exists():
    urls = [
        'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt',
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov5s.pt',
    ]
    ok = False
    for u in urls:
        r = subprocess.run(['wget','-q','--show-progress','-O',str(YOLO_WEIGHTS),u])
        if r.returncode == 0 and YOLO_WEIGHTS.exists() and YOLO_WEIGHTS.stat().st_size > 1_000_000:
            ok = True
            break
    assert ok, 'Không tải được yolov5s.pt. Kiểm tra Internet của Kaggle.'

print('DCVC-RT:', DCVC_RT)
print('YOLO repo:', YOLO_REPO)
print('YOLO weights:', YOLO_WEIGHTS, YOLO_WEIGHTS.stat().st_size/1024**2, 'MB')
assert (DCVC_RT/'src/models/video_model.py').exists()
assert (DCVC_RT/'src/models/image_model.py').exists()

In [ ]:
# 5) Sinh training/evaluation script dùng bởi torchrun (DDP 2 GPU)
from pathlib import Path
TRAIN_SCRIPT = Path('/kaggle/working/train_dcvc_rt_vcm_ddp.py')
TRAIN_SCRIPT.write_text("#!/usr/bin/env python3\nimport argparse\nimport contextlib\nimport copy\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom PIL import Image\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument('--dcvc_rt', required=True)\n    p.add_argument('--yolo_repo', required=True)\n    p.add_argument('--yolo_weights', required=True)\n    p.add_argument('--vimeo_root', required=True)\n    p.add_argument('--image_ckpt', required=True)\n    p.add_argument('--video_ckpt', required=True)\n    p.add_argument('--output_dir', required=True)\n    p.add_argument('--lambda_min', type=float, default=1.0)\n    p.add_argument('--lambda_max', type=float, default=64.0)\n    p.add_argument('--epochs', type=int, default=40)\n    p.add_argument('--early_stopping_patience', type=int, default=5)\n    p.add_argument('--early_stopping_min_delta', type=float, default=1e-4)\n    p.add_argument('--lr_video', type=float, default=1e-6)\n    p.add_argument('--lr_yolo', type=float, default=1e-6)\n    p.add_argument('--weight_decay', type=float, default=0.0)\n    p.add_argument('--crop_size', type=int, default=256)\n    p.add_argument('--p_frames', type=int, default=6)\n    p.add_argument('--batch_per_gpu', type=int, default=2)\n    p.add_argument('--grad_accum', type=int, default=1)\n    p.add_argument('--num_workers', type=int, default=2)\n    p.add_argument('--qp_min', type=int, default=0)\n    p.add_argument('--qp_max', type=int, default=63)\n    p.add_argument('--grad_clip', type=float, default=1.0)\n    p.add_argument('--amp', type=int, default=1)\n    p.add_argument('--seed', type=int, default=1234)\n    p.add_argument('--max_train_samples', type=int, default=0)\n    p.add_argument('--max_val_samples', type=int, default=400)\n    p.add_argument('--max_steps', type=int, default=0)\n    p.add_argument('--val_qps', type=str, default='8,24,40,56')\n    p.add_argument('--eval_qps', type=str, default='0,8,16,24,32,40,48,56,63')\n    p.add_argument('--resume', type=str, default='')\n    p.add_argument('--model_ckpt', type=str, default='')\n    p.add_argument('--eval_only', type=int, default=0)\n    p.add_argument('--eval_baseline', type=int, default=0)\n    p.add_argument('--comet', type=int, default=0)\n    p.add_argument('--comet_project', default='dcvc-rt-vcm')\n    return p.parse_args()\n\n\ndef setup_ddp(seed):\n    dist.init_process_group('nccl')\n    rank = dist.get_rank()\n    world = dist.get_world_size()\n    local_rank = int(os.environ['LOCAL_RANK'])\n    torch.cuda.set_device(local_rank)\n    device = torch.device('cuda', local_rank)\n    random.seed(seed + rank)\n    np.random.seed(seed + rank)\n    torch.manual_seed(seed + rank)\n    torch.cuda.manual_seed_all(seed + rank)\n    return rank, world, local_rank, device\n\n\ndef is_main():\n    return dist.get_rank() == 0\n\n\ndef cleanup():\n    dist.barrier()\n    dist.destroy_process_group()\n\n\nQP_OFFSETS = (0, 8, 0, 4, 0, 4, 0, 4)\nDISTORTION_WEIGHTS = (0.5, 1.2, 0.5, 0.9)\n\n\ndef lambda_for_qp(qp, args):\n    if not args.qp_min <= qp <= args.qp_max:\n        raise ValueError(f'QP {qp} is outside [{args.qp_min}, {args.qp_max}]')\n    position = (qp - args.qp_min) / (args.qp_max - args.qp_min)\n    return args.lambda_min * (args.lambda_max / args.lambda_min) ** position\n\n\ndef synchronized_random_qp(args, device):\n    qp = random.randint(args.qp_min, args.qp_max) if is_main() else 0\n    value = torch.tensor(qp, device=device)\n    dist.broadcast(value, 0)\n    return int(value.item())\n\n\nclass VimeoSeptuplet(Dataset):\n    def __init__(self, root, split='train', crop=256, p_frames=6, max_samples=0):\n        self.root = Path(root)\n        self.seq_root = self.root / 'sequences'\n        split_file = self.root / ('sep_trainlist.txt' if split == 'train' else 'sep_testlist.txt')\n        if not split_file.exists():\n            raise FileNotFoundError(\n                f'Missing required split file: {split_file}. '\n                'Validation must not silently fall back to the training list.'\n            )\n        self.items = [x.strip() for x in split_file.read_text().splitlines() if x.strip()]\n        if max_samples and max_samples > 0:\n            self.items = self.items[:max_samples]\n        self.train = split == 'train'\n        self.crop = crop\n        self.num_frames = 1 + p_frames\n        assert self.num_frames <= 7\n\n    def __len__(self):\n        return len(self.items)\n\n    @staticmethod\n    def load_rgb(path):\n        with Image.open(path) as im:\n            arr = np.asarray(im.convert('RGB'), dtype=np.uint8).copy()\n        return torch.from_numpy(arr).permute(2, 0, 1).float().div_(255.0)\n\n    def __getitem__(self, idx):\n        d = self.seq_root / self.items[idx]\n        frames = [self.load_rgb(d / f'im{i}.png') for i in range(1, self.num_frames + 1)]\n        _, H, W = frames[0].shape\n        c = self.crop\n        if H < c or W < c:\n            raise RuntimeError(f'Frame smaller than crop: {d} -> {H}x{W}, crop={c}')\n        if self.train:\n            top = random.randint(0, H - c)\n            left = random.randint(0, W - c)\n            flip = random.random() < 0.5\n        else:\n            top = (H - c) // 2\n            left = (W - c) // 2\n            flip = False\n        out = []\n        for x in frames:\n            x = x[:, top:top+c, left:left+c]\n            if flip:\n                x = torch.flip(x, dims=[2])\n            out.append(x)\n        return torch.stack(out, 0)  # [T,3,H,W]\n\n\ndef load_yolo_front(yolo_repo, weights, device, n_layers=5):\n    sys.path.insert(0, str(yolo_repo))\n    # torch>=2.6 defaults to weights_only=True; YOLOv5 .pt stores a model object.\n    ckpt = torch.load(weights, map_location='cpu', weights_only=False)\n    core = ckpt.get('ema', None)\n    if core is None:\n        core = ckpt['model']\n    core = core.float().eval()\n    seq = core.model\n    teacher = nn.Sequential(*[copy.deepcopy(seq[i]) for i in range(n_layers)]).to(device).eval()\n    student = nn.Sequential(*[copy.deepcopy(seq[i]) for i in range(n_layers)]).to(device)\n    for p in teacher.parameters():\n        p.requires_grad_(False)\n    return teacher, student\n\n\ndef freeze_bn(module):\n    for m in module.modules():\n        if isinstance(m, nn.BatchNorm2d):\n            m.eval()\n            for p in m.parameters():\n                p.requires_grad_(False)\n\n\ndef ste_round(x):\n    return x + (torch.round(x) - x).detach()\n\n\ndef normal_cdf(x):\n    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))\n\n\ndef gaussian_bits(symbols, scales, mask):\n    # Match DCVC-RT GaussianEncoder coding range: scale_min=.11, scale_max=16.\n    with torch.autocast(device_type='cuda', enabled=False):\n        s = scales.float().clamp(0.11, 16.0)\n        y = symbols.float()\n        upper = normal_cdf((y + 0.5) / s)\n        lower = normal_cdf((y - 0.5) / s)\n        prob = (upper - lower).clamp_min(1e-9)\n        bits = -torch.log2(prob)\n        return bits * mask.float()\n\n\ndef z_bits(model, z_hat, qp):\n    B = z_hat.shape[0]\n    idx = torch.full((B,), int(qp), dtype=torch.long, device=z_hat.device)\n    with torch.autocast(device_type='cuda', enabled=False):\n        z = z_hat.float()\n        upper = model.bit_estimator_z(z + 0.5, idx)\n        lower = model.bit_estimator_z(z - 0.5, idx)\n        prob = (upper - lower).clamp_min(1e-9)\n        return -torch.log2(prob)\n\n\ndef masked_quant_train(y, scales, means, mask):\n    scales_hat = scales * mask\n    means_hat = means * mask\n    y_res = (y - means_hat) * mask\n    y_q = ste_round(y_res).clamp(-128.0, 127.0)\n    y_hat = y_q + means_hat\n    return y_q, y_hat, scales_hat\n\n\nclass VCMSystem(nn.Module):\n    INDEX_MAP = [0, 1, 0, 2, 0, 2, 0, 2]\n    DISTORTION_WEIGHTS = DISTORTION_WEIGHTS\n\n    def __init__(self, p_net, student_front):\n        super().__init__()\n        self.p_net = p_net\n        self.student_front = student_front\n\n    def p_frame_forward(self, x, qp):\n        m = self.p_net\n        q_encoder = m.q_encoder[qp:qp+1]\n        q_decoder = m.q_decoder[qp:qp+1]\n        q_feature = m.q_feature[qp:qp+1]\n        q_recon = m.q_recon[qp:qp+1]\n\n        ref_feature = m.apply_feature_adaptor()\n        ctx, ctx_t = m.feature_extractor(ref_feature, q_feature)\n        y = m.encoder(x, ctx, q_encoder)\n\n        hyper_inp = m.pad_for_y(y)\n        z = m.hyper_encoder(hyper_inp)\n        z_hat = ste_round(z).clamp(-128.0, 127.0)\n\n        common_params = m.res_prior_param_decoder(z_hat, ctx_t)\n        y_scaled, q_dec, scales, means = m.separate_prior_for_video_encoding(common_params, y)\n        B, C, H_y, W_y = y_scaled.shape\n        mask0, mask1 = m.get_mask_2x(B, C, H_y, W_y, y_scaled.dtype, y_scaled.device)\n\n        yq0, yh0, s0 = masked_quant_train(y_scaled, scales, means, mask0)\n        cat_params = torch.cat((yh0, common_params), dim=1)\n        scales1, means1 = m.y_spatial_prior(cat_params).chunk(2, 1)\n        yq1, yh1, s1 = masked_quant_train(y_scaled, scales1, means1, mask1)\n\n        y_hat = (yh0 + yh1) * q_dec\n        x_hat, feature = m.get_recon_and_feature(y_hat, ctx, q_decoder, q_recon)\n\n        by0 = gaussian_bits(yq0, s0, mask0).flatten(1).sum(1)\n        by1 = gaussian_bits(yq1, s1, mask1).flatten(1).sum(1)\n        bz = z_bits(m, z_hat, qp).flatten(1).sum(1)\n        pixel_num = x.shape[-2] * x.shape[-1]\n        bpp = (by0 + by1 + bz) / pixel_num\n\n        # Keep temporal graph across P frames (BPTT over this short clip).\n        m.add_ref_frame(feature=feature, frame=x_hat)\n        return x_hat, bpp\n\n    def forward(self, x0_hat_yuv, p_yuv, p_rgb, teacher_feats, base_qp, lambda_machine):\n        m = self.p_net\n        m.clear_dpb()\n        m.set_curr_poc(0)\n        m.add_ref_frame(feature=None, frame=x0_hat_yuv)\n\n        loss_terms = []\n        rate_terms = []\n        feat_terms = []\n        psnr_terms = []\n\n        T = p_yuv.shape[1]\n        for t in range(T):\n            frame_idx = t + 1\n            fa_idx = self.INDEX_MAP[frame_idx % 8]\n            curr_qp = m.shift_qp(int(base_qp), fa_idx)\n            x_hat_yuv, bpp_vec = self.p_frame_forward(p_yuv[:, t], curr_qp)\n            x_hat_rgb = ycbcr2rgb(x_hat_yuv, clamp=True)\n            feat_hat = self.student_front(x_hat_rgb)\n            feat_loss = F.mse_loss(feat_hat.float(), teacher_feats[:, t].float(), reduction='mean')\n            rate = bpp_vec.mean()\n            mse_rgb = F.mse_loss(x_hat_rgb.float(), p_rgb[:, t].float(), reduction='mean')\n            psnr = -10.0 * torch.log10(mse_rgb.clamp_min(1e-12))\n            distortion_weight = self.DISTORTION_WEIGHTS[t % len(self.DISTORTION_WEIGHTS)]\n            loss_terms.append(rate + float(lambda_machine) * distortion_weight * feat_loss)\n            rate_terms.append(rate)\n            feat_terms.append(feat_loss)\n            psnr_terms.append(psnr)\n\n        return {\n            'loss': torch.stack(loss_terms).mean(),\n            'bpp': torch.stack(rate_terms).mean(),\n            'feature_mse': torch.stack(feat_terms).mean(),\n            'psnr': torch.stack(psnr_terms).mean(),\n        }\n\n\ndef intra_reconstruct(i_net, x_yuv, qp):\n    # Frozen reconstruction-only path; no entropy-coder extension needed.\n    curr_q_enc = i_net.q_scale_enc[qp:qp+1]\n    curr_q_dec = i_net.q_scale_dec[qp:qp+1]\n    y = i_net.enc(x_yuv, curr_q_enc)\n    y_pad = i_net.pad_for_y(y)\n    z = i_net.hyper_enc(y_pad)\n    z_hat = torch.round(z).clamp(-128.0, 127.0)\n    params = i_net.hyper_dec(z_hat)\n    params = i_net.y_prior_fusion(params)\n    _, _, yH, yW = y.shape\n    params = params[:, :, :yH, :yW].contiguous()\n    *_, y_hat = i_net.compress_prior_4x(\n        y, params,\n        i_net.y_spatial_prior_reduction,\n        i_net.y_spatial_prior_adaptor_1,\n        i_net.y_spatial_prior_adaptor_2,\n        i_net.y_spatial_prior_adaptor_3,\n        i_net.y_spatial_prior,\n    )\n    return i_net.dec(y_hat, curr_q_dec).clamp(0.0, 1.0)\n\n\ndef reduce_metrics(sums, count, device):\n    vec = torch.tensor([sums['loss'], sums['bpp'], sums['feature_mse'], sums['psnr'], count],\n                       device=device, dtype=torch.float64)\n    dist.all_reduce(vec, op=dist.ReduceOp.SUM)\n    n = max(float(vec[4].item()), 1.0)\n    return {\n        'loss': float(vec[0].item()/n),\n        'bpp': float(vec[1].item()/n),\n        'feature_mse': float(vec[2].item()/n),\n        'psnr': float(vec[3].item()/n),\n    }\n\n\ndef make_loaders(args, rank, world):\n    train_ds = VimeoSeptuplet(args.vimeo_root, 'train', args.crop_size, args.p_frames,\n                              args.max_train_samples)\n    val_ds = VimeoSeptuplet(args.vimeo_root, 'val', args.crop_size, args.p_frames,\n                            args.max_val_samples)\n    train_sampler = DistributedSampler(train_ds, num_replicas=world, rank=rank, shuffle=True, drop_last=True)\n    val_sampler = DistributedSampler(val_ds, num_replicas=world, rank=rank, shuffle=False, drop_last=False)\n    common = dict(batch_size=args.batch_per_gpu, num_workers=args.num_workers,\n                  pin_memory=True, persistent_workers=args.num_workers > 0)\n    train_loader = DataLoader(train_ds, sampler=train_sampler, drop_last=True, **common)\n    val_loader = DataLoader(val_ds, sampler=val_sampler, drop_last=False, **common)\n    return train_loader, val_loader, train_sampler, train_ds, val_ds\n\n\ndef prepare_batch(frames, device, i_net, teacher, base_qp, use_amp):\n    frames = frames.to(device, non_blocking=True)  # [B,1+T,3,H,W] RGB\n    x0_rgb = frames[:, 0]\n    p_rgb = frames[:, 1:]\n    with torch.no_grad():\n        x0_yuv = rgb2ycbcr(x0_rgb)\n        p_yuv = torch.stack([rgb2ycbcr(p_rgb[:, t]) for t in range(p_rgb.shape[1])], dim=1)\n        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):\n            x0_hat = intra_reconstruct(i_net, x0_yuv, int(base_qp))\n            feats = [teacher(p_rgb[:, t]) for t in range(p_rgb.shape[1])]\n        teacher_feats = torch.stack(feats, dim=1)\n    return x0_hat, p_yuv, p_rgb, teacher_feats\n\n\n@torch.no_grad()\ndef validate(system, i_net, teacher, loader, device, args, fixed_qp=None):\n    system.eval()\n    freeze_bn(system.module.student_front)\n    sums = {'loss':0.0, 'bpp':0.0, 'feature_mse':0.0, 'psnr':0.0}\n    count = 0\n    val_qps = [int(x) for x in args.val_qps.split(',') if x.strip()]\n    for bi, frames in enumerate(loader):\n        qp = int(fixed_qp if fixed_qp is not None else val_qps[bi % len(val_qps)])\n        x0_hat, p_yuv, p_rgb, teacher_feats = prepare_batch(frames, device, i_net, teacher, qp, bool(args.amp))\n        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=bool(args.amp)):\n            out = system(x0_hat, p_yuv, p_rgb, teacher_feats, qp, lambda_for_qp(qp, args))\n        bs = frames.shape[0]\n        for k in sums:\n            sums[k] += float(out[k].detach().item()) * bs\n        count += bs\n    return reduce_metrics(sums, count, device)\n\n\ndef save_epoch_plot(history_path, output_path):\n    rows = [\n        json.loads(line)\n        for line in history_path.read_text(encoding='utf-8').splitlines()\n        if line.strip()\n    ]\n    if not rows:\n        return\n    epochs = [row.get('epoch_number', row['epoch'] + 1) for row in rows]\n    figure, axes = plt.subplots(1, 3, figsize=(15, 4.2))\n    for axis, metric, title in zip(\n        axes,\n        ('loss', 'bpp', 'feature_mse'),\n        ('Total Loss', 'BPP', 'Feature MSE'),\n    ):\n        axis.plot(epochs, [row['train'][metric] for row in rows], 'o-', label='Train')\n        axis.plot(epochs, [row['val'][metric] for row in rows], 's--', label='Validation')\n        axis.set(xlabel='Epoch', ylabel=title, title=title)\n        axis.grid(alpha=0.3)\n        axis.legend()\n    figure.suptitle('Random QP 0..63 — mapped lambda 1..64')\n    figure.tight_layout()\n    figure.savefig(output_path, dpi=180, bbox_inches='tight')\n    plt.close(figure)\n\n\ndef main():\n    args = parse_args()\n    if not 0 <= args.qp_min < args.qp_max <= 63:\n        raise ValueError('QP range must satisfy 0 <= qp_min < qp_max <= 63')\n    if not 0 < args.lambda_min < args.lambda_max:\n        raise ValueError('Lambda range must satisfy 0 < lambda_min < lambda_max')\n    rank, world, local_rank, device = setup_ddp(args.seed)\n\n    sys.path.insert(0, args.dcvc_rt)\n    global DMC, DMCI, get_state_dict, rgb2ycbcr, ycbcr2rgb\n    from src.models.video_model import DMC\n    from src.models.image_model import DMCI\n    from src.utils.common import get_state_dict\n    from src.utils.transforms import rgb2ycbcr, ycbcr2rgb\n\n    out_dir = Path(args.output_dir)\n    if is_main():\n        out_dir.mkdir(parents=True, exist_ok=True)\n        (out_dir/'config.json').write_text(json.dumps(vars(args), indent=2))\n\n    experiment = None\n    if is_main() and args.comet:\n        try:\n            import comet_ml\n            experiment = comet_ml.start(\n                project_name=args.comet_project,\n                experiment_config=comet_ml.ExperimentConfig(\n                    name=out_dir.name,\n                    parse_args=False,\n                ),\n            )\n            experiment.log_parameters(vars(args))\n            print('COMET: enabled', flush=True)\n        except Exception as exc:\n            print('COMET DISABLED:', repr(exc), flush=True)\n\n    teacher, student = load_yolo_front(args.yolo_repo, args.yolo_weights, device, 5)\n    freeze_bn(student)\n\n    i_net = DMCI().to(device)\n    i_net.load_state_dict(get_state_dict(args.image_ckpt), strict=True)\n    i_net.eval()\n    for p in i_net.parameters(): p.requires_grad_(False)\n\n    p_net = DMC().to(device)\n    p_net.load_state_dict(get_state_dict(args.video_ckpt), strict=True)\n    actual_offsets = tuple(p_net.qp_shift[index] for index in VCMSystem.INDEX_MAP)\n    if actual_offsets != QP_OFFSETS:\n        raise RuntimeError(f'DCVC-RT QP offsets are {actual_offsets}, expected {QP_OFFSETS}')\n\n    start_epoch = 0\n    best_val = float('inf')\n    bad_epochs = 0\n    resume_obj = None\n    model_ckpt = args.model_ckpt or args.resume\n    if model_ckpt:\n        resume_obj = torch.load(model_ckpt, map_location='cpu', weights_only=False)\n        if 'p_net' in resume_obj:\n            p_net.load_state_dict(resume_obj['p_net'], strict=True)\n        if 'student_front' in resume_obj:\n            student.load_state_dict(resume_obj['student_front'], strict=True)\n        if args.resume:\n            saved_args = resume_obj.get('args', {})\n            for key in ('qp_min', 'qp_max', 'lambda_min', 'lambda_max', 'p_frames'):\n                if float(saved_args.get(key, float('nan'))) != float(getattr(args, key)):\n                    raise RuntimeError(f'Resume checkpoint uses a different {key}')\n            if tuple(resume_obj.get('qp_offsets', ())) != QP_OFFSETS:\n                raise RuntimeError('Resume checkpoint uses different hierarchical QP offsets')\n            if tuple(resume_obj.get('distortion_weights', ())) != DISTORTION_WEIGHTS:\n                raise RuntimeError('Resume checkpoint uses different distortion weights')\n            if not bool(resume_obj.get('epoch_complete', False)):\n                raise RuntimeError(\n                    'Refusing to resume a legacy/partial checkpoint because epoch_complete=True '\n                    'is not recorded. Start from a clean output directory or a verified full-epoch checkpoint.'\n                )\n            start_epoch = int(resume_obj.get('epoch', -1)) + 1\n            best_val = float(resume_obj.get('best_val', best_val))\n            bad_epochs = int(resume_obj.get('bad_epochs', 0))\n\n    system_raw = VCMSystem(p_net, student).to(device)\n    system = DDP(system_raw, device_ids=[local_rank], output_device=local_rank,\n                 broadcast_buffers=False, find_unused_parameters=False)\n\n    train_loader, val_loader, train_sampler, train_ds, val_ds = make_loaders(args, rank, world)\n\n    # Ground-truth dataset accounting for THIS attached Kaggle dataset.\n    if is_main():\n        dataset_info = {\n            'train_list_sequences': len(train_ds),\n            'val_list_sequences': len(val_ds),\n            'world_size': world,\n            'batch_per_gpu': args.batch_per_gpu,\n            'grad_accum': args.grad_accum,\n            'p_frames': args.p_frames,\n            'frames_loaded_per_sequence': 1 + args.p_frames,\n            'train_sampler_samples_per_rank': int(train_sampler.num_samples),\n            'train_sampler_total_size': int(train_sampler.total_size),\n            'train_loader_batches_per_rank': len(train_loader),\n            'effective_train_sequences_per_full_epoch': (\n                len(train_loader) * args.batch_per_gpu * world\n            ),\n            'note': (\n                'max_train_samples=0 means all lines in the attached sep_trainlist.txt; '\n                'it does not imply a particular official Vimeo-90K split size.'\n            ),\n        }\n        (out_dir/'dataset_info.json').write_text(\n            json.dumps(dataset_info, indent=2), encoding='utf-8'\n        )\n        print('DATASET INFO:', json.dumps(dataset_info), flush=True)\n\n    if args.eval_only:\n        qps = [int(x) for x in args.eval_qps.split(',') if x.strip()]\n        rows = []\n        for qp in qps:\n            metrics = validate(system, i_net, teacher, val_loader, device, args, fixed_qp=qp)\n            if is_main():\n                row = {'base_qp':qp, **metrics}\n                rows.append(row)\n                print('EVAL', row, flush=True)\n        if is_main():\n            (out_dir/'eval_sweep.json').write_text(json.dumps(rows, indent=2))\n            with open(out_dir/'eval_sweep.csv','w',newline='') as f:\n                w = csv.DictWriter(f, fieldnames=rows[0].keys())\n                w.writeheader(); w.writerows(rows)\n        if experiment is not None:\n            experiment.end()\n        cleanup(); return\n\n    params_video = [p for p in system.module.p_net.parameters() if p.requires_grad]\n    params_yolo = [p for p in system.module.student_front.parameters() if p.requires_grad]\n    optimizer = torch.optim.Adam([\n        {'params':params_video, 'lr':args.lr_video},\n        {'params':params_yolo, 'lr':args.lr_yolo},\n    ], weight_decay=args.weight_decay)\n    scaler = torch.amp.GradScaler('cuda', enabled=bool(args.amp))\n\n    if args.resume and resume_obj is not None:\n        if 'optimizer' in resume_obj: optimizer.load_state_dict(resume_obj['optimizer'])\n        if 'scaler' in resume_obj: scaler.load_state_dict(resume_obj['scaler'])\n\n    history_path = out_dir/'history.jsonl'\n    history_csv_path = out_dir/'history.csv'\n    epoch_metrics_dir = out_dir/'epoch_metrics'\n    epoch_plots_dir = out_dir/'epoch_plots'\n\n    global_step = int(resume_obj.get('global_step', 0)) if (args.resume and resume_obj is not None) else 0\n    optimizer_step = int(resume_obj.get('optimizer_step', 0)) if (args.resume and resume_obj is not None) else 0\n\n    if is_main():\n        epoch_metrics_dir.mkdir(parents=True, exist_ok=True)\n        epoch_plots_dir.mkdir(parents=True, exist_ok=True)\n        if not args.resume:\n            # Fresh run: never append to stale histories from an older experiment.\n            history_path.write_text('', encoding='utf-8')\n            if history_csv_path.exists():\n                history_csv_path.unlink()\n            for p in epoch_metrics_dir.glob('epoch_*.json'):\n                p.unlink()\n\n    dist.barrier()\n    optimizer.zero_grad(set_to_none=True)\n\n    for epoch in range(start_epoch, args.epochs):\n        train_sampler.set_epoch(epoch)\n        system.train()\n        freeze_bn(system.module.student_front)\n        sums = {'loss':0.0, 'bpp':0.0, 'feature_mse':0.0, 'psnr':0.0}\n        count = 0\n        epoch_steps = 0\n        epoch_optimizer_steps = 0\n        epoch_start_time = time.time()\n\n        for it, frames in enumerate(train_loader):\n            base_qp = synchronized_random_qp(args, device)\n            lambda_machine = lambda_for_qp(base_qp, args)\n            x0_hat, p_yuv, p_rgb, teacher_feats = prepare_batch(frames, device, i_net, teacher, base_qp, bool(args.amp))\n            # Correct gradient normalization for the final incomplete accumulation group.\n            remainder = len(train_loader) % args.grad_accum\n            in_final_remainder = remainder > 0 and it >= (len(train_loader) - remainder)\n            accum_divisor = remainder if in_final_remainder else args.grad_accum\n\n            do_step = ((it + 1) % args.grad_accum == 0) or (it + 1 == len(train_loader))\n            sync_ctx = contextlib.nullcontext() if do_step else system.no_sync()\n            with sync_ctx:\n                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=bool(args.amp)):\n                    out = system(x0_hat, p_yuv, p_rgb, teacher_feats, base_qp, lambda_machine)\n                    scaled_loss = out['loss'] / float(accum_divisor)\n                scaler.scale(scaled_loss).backward()\n\n            if do_step:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(system.parameters(), args.grad_clip)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                optimizer_step += 1\n                epoch_optimizer_steps += 1\n\n            bs = frames.shape[0]\n            for k in sums:\n                sums[k] += float(out[k].detach().item()) * bs\n            count += bs\n            global_step += 1\n            epoch_steps += 1\n\n            if is_main() and global_step % 20 == 0:\n                print(f\"epoch={epoch} step={global_step} qp={base_qp} \"\n                      f\"loss={out['loss'].item():.6f} bpp={out['bpp'].item():.6f} \"\n                      f\"feat={out['feature_mse'].item():.6e} lambda={lambda_machine:.6f} \"\n                      f\"psnr={out['psnr'].item():.3f}\", flush=True)\n\n            if args.max_steps and global_step >= args.max_steps:\n                break\n\n        train_m = reduce_metrics(sums, count, device)\n        val_m = validate(system, i_net, teacher, val_loader, device, args)\n\n        epoch_complete = (epoch_steps == len(train_loader))\n        epoch_elapsed = time.time() - epoch_start_time\n\n        if is_main():\n            improved = val_m['loss'] < best_val - args.early_stopping_min_delta\n            if improved:\n                best_val = val_m['loss']\n                bad_epochs = 0\n            else:\n                bad_epochs += 1\n            should_stop = (\n                epoch_complete\n                and bad_epochs >= args.early_stopping_patience\n            )\n\n            rec = {\n                'epoch': epoch,\n                'epoch_number': epoch + 1,\n                'epoch_complete': epoch_complete,\n                'global_step': global_step,\n                'steps_this_epoch': epoch_steps,\n                'expected_steps_full_epoch': len(train_loader),\n                'optimizer_step': optimizer_step,\n                'optimizer_updates_this_epoch': epoch_optimizer_steps,\n                'samples_seen_global_this_epoch': epoch_steps * args.batch_per_gpu * world,\n                'lambda_range': [args.lambda_min, args.lambda_max],\n                'lambda_mapping': 'lambda(q)=lambda_min*(lambda_max/lambda_min)**((q-qp_min)/(qp_max-qp_min))',\n                'qp_sampling': [args.qp_min, args.qp_max],\n                'qp_offsets': list(QP_OFFSETS),\n                'distortion_weights': list(DISTORTION_WEIGHTS),\n                'elapsed_sec': epoch_elapsed,\n                'best_val_loss': best_val,\n                'early_stopping_bad_epochs': bad_epochs,\n                'early_stopping_patience': args.early_stopping_patience,\n                'early_stopped': should_stop,\n                'train': train_m,\n                'val': val_m,\n            }\n\n            with open(history_path, 'a', encoding='utf-8') as f:\n                f.write(json.dumps(rec) + '\\n')\n\n            flat = {\n                'lambda_range': [args.lambda_min, args.lambda_max],\n                'lambda_mapping': 'lambda(q)=lambda_min*(lambda_max/lambda_min)**((q-qp_min)/(qp_max-qp_min))',\n                'qp_sampling': [args.qp_min, args.qp_max],\n                'qp_offsets': list(QP_OFFSETS),\n                'distortion_weights': list(DISTORTION_WEIGHTS),\n                'epoch': epoch,\n                'epoch_number': epoch + 1,\n                'epoch_complete': epoch_complete,\n                'global_step': global_step,\n                'steps_this_epoch': epoch_steps,\n                'expected_steps_full_epoch': len(train_loader),\n                'optimizer_step': optimizer_step,\n                'optimizer_updates_this_epoch': epoch_optimizer_steps,\n                'samples_seen_global_this_epoch': epoch_steps * args.batch_per_gpu * world,\n                'elapsed_sec': epoch_elapsed,\n                'best_val_loss': best_val,\n                'early_stopping_bad_epochs': bad_epochs,\n                'early_stopped': should_stop,\n                'train_loss': train_m['loss'],\n                'train_bpp': train_m['bpp'],\n                'train_feature_mse': train_m['feature_mse'],\n                'train_psnr': train_m['psnr'],\n                'val_loss': val_m['loss'],\n                'val_bpp': val_m['bpp'],\n                'val_feature_mse': val_m['feature_mse'],\n                'val_psnr': val_m['psnr'],\n            }\n            csv_exists = history_csv_path.exists() and history_csv_path.stat().st_size > 0\n            with open(history_csv_path, 'a', newline='', encoding='utf-8') as f:\n                writer = csv.DictWriter(f, fieldnames=list(flat.keys()))\n                if not csv_exists:\n                    writer.writeheader()\n                writer.writerow(flat)\n\n            epoch_json = epoch_metrics_dir / f'epoch_{epoch + 1:02d}.json'\n            epoch_json.write_text(json.dumps(rec, indent=2), encoding='utf-8')\n            epoch_plot = epoch_plots_dir / f'epoch_{epoch + 1:02d}_loss_bpp_feature_mse.png'\n            save_epoch_plot(history_path, epoch_plot)\n\n            if experiment is not None:\n                experiment.log_metrics({\n                    'train/loss': train_m['loss'],\n                    'train/bpp': train_m['bpp'],\n                    'train/feature_mse': train_m['feature_mse'],\n                    'train/psnr': train_m['psnr'],\n                    'val/loss': val_m['loss'],\n                    'val/bpp': val_m['bpp'],\n                    'val/feature_mse': val_m['feature_mse'],\n                    'val/psnr': val_m['psnr'],\n                    'early_stopping/best_val_loss': best_val,\n                    'early_stopping/bad_epochs': bad_epochs,\n                }, step=epoch + 1)\n                experiment.log_image(\n                    image_data=str(epoch_plot),\n                    name='training_curves.png',\n                    step=epoch + 1,\n                )\n\n            print('EPOCH SUMMARY:', json.dumps(rec), flush=True)\n            print('SAVED EPOCH METRICS:', epoch_json, flush=True)\n            print('SAVED EPOCH PLOT:', epoch_plot, flush=True)\n\n            core = system.module\n            obj = {\n                'p_net': core.p_net.state_dict(),\n                'student_front': core.student_front.state_dict(),\n                'optimizer': optimizer.state_dict(),\n                'scaler': scaler.state_dict(),\n                'epoch': epoch,\n                'epoch_complete': epoch_complete,\n                'global_step': global_step,\n                'optimizer_step': optimizer_step,\n                'best_val': best_val,\n                'bad_epochs': bad_epochs,\n                'early_stopped': should_stop,\n                'schema_version': 7,\n                'lambda_task': None,\n                'hierarchical_qp': True,\n                'hierarchical_qp_offsets': list(QP_OFFSETS),\n                'hierarchical_distortion_weights': list(DISTORTION_WEIGHTS),\n                'lambda_range': [args.lambda_min, args.lambda_max],\n                'lambda_mapping': 'lambda(q)=lambda_min*(lambda_max/lambda_min)**((q-qp_min)/(qp_max-qp_min))',\n                'qp_sampling': [args.qp_min, args.qp_max],\n                'qp_offsets': list(QP_OFFSETS),\n                'distortion_weights': list(DISTORTION_WEIGHTS),\n                'args': vars(args),\n            }\n            torch.save(obj, out_dir/'last.pth.tar')\n            if improved:\n                torch.save(obj, out_dir/'best.pth.tar')\n                print('Saved new best:', best_val, flush=True)\n            if should_stop:\n                print(\n                    f'EARLY STOP: val loss did not improve by '\n                    f'{args.early_stopping_min_delta:g} for '\n                    f'{args.early_stopping_patience} epochs.',\n                    flush=True,\n                )\n\n        stop_flag = torch.tensor(\n            1 if is_main() and should_stop else 0,\n            device=device,\n            dtype=torch.int32,\n        )\n        dist.broadcast(stop_flag, 0)\n        dist.barrier()\n        if stop_flag.item() or (args.max_steps and global_step >= args.max_steps):\n            break\n\n    if experiment is not None:\n        experiment.end()\n    cleanup()\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
print('Wrote:', TRAIN_SCRIPT)
print('Size:', TRAIN_SCRIPT.stat().st_size, 'bytes')


In [ ]:
# 6) Helper tạo lệnh torchrun 2 GPU
import subprocess
import shlex
from pathlib import Path

def make_cmd(
    output_dir,
    epochs=None,
    max_steps=0,
    max_train_samples=None,
    eval_only=False,
    model_ckpt='',
    eval_baseline=False,
    resume='',
):
    cmd = [
        'torchrun', '--standalone', '--nproc_per_node=2', str(TRAIN_SCRIPT),
        '--dcvc_rt', str(DCVC_RT),
        '--yolo_repo', str(YOLO_REPO),
        '--yolo_weights', str(YOLO_WEIGHTS),
        '--vimeo_root', str(VIMEO_ROOT),
        '--image_ckpt', str(IMAGE_CKPT),
        '--video_ckpt', str(VIDEO_CKPT),
        '--output_dir', str(output_dir),
        '--lambda_min', str(CFG['lambda_min']),
        '--lambda_max', str(CFG['lambda_max']),
        '--epochs', str(CFG['epochs'] if epochs is None else epochs),
        '--early_stopping_patience', str(CFG['early_stopping_patience']),
        '--early_stopping_min_delta', str(CFG['early_stopping_min_delta']),
        '--lr_video', str(CFG['lr_video']),
        '--lr_yolo', str(CFG['lr_yolo']),
        '--weight_decay', str(CFG['weight_decay']),
        '--crop_size', str(CFG['crop_size']),
        '--p_frames', str(CFG['p_frames']),
        '--batch_per_gpu', str(CFG['batch_per_gpu']),
        '--grad_accum', str(CFG['grad_accum']),
        '--num_workers', str(CFG['num_workers']),
        '--qp_min', str(CFG['qp_min']),
        '--qp_max', str(CFG['qp_max']),
        '--grad_clip', str(CFG['grad_clip']),
        '--amp', '1' if CFG['amp'] else '0',
        '--seed', str(CFG['seed']),
        '--max_train_samples', str(
            CFG['max_train_samples'] if max_train_samples is None
            else max_train_samples
        ),
        '--max_val_samples', str(CFG['max_val_samples']),
        '--max_steps', str(max_steps),
        '--val_qps', CFG['val_qps'],
        '--eval_qps', CFG['eval_qps'],
        '--eval_only', '1' if eval_only else '0',
        '--eval_baseline', '1' if eval_baseline else '0',
        '--comet', '1' if USE_COMET else '0',
        '--comet_project', COMET_PROJECT_NAME,
    ]
    if model_ckpt:
        cmd += ['--model_ckpt', str(model_ckpt)]
    if resume:
        cmd += ['--resume', str(resume)]
    return cmd

print('Example smoke command:')
print(' '.join(shlex.quote(x) for x in make_cmd(
    '/kaggle/working/smoke_random_qp_lambda_1_64',
    epochs=1,
    max_steps=2,
    max_train_samples=16,
)))


In [ ]:
# 7) SMOKE TEST — 2 GPU, chỉ 2 step
from pathlib import Path
import subprocess

if RUN_SMOKE_TEST:
    smoke_dir = Path(CFG['output_root']) / 'smoke'
    subprocess.run(make_cmd(
        smoke_dir,
        epochs=1,
        max_steps=2,
        max_train_samples=16,
    ), check=True)
    print('Smoke test OK:', smoke_dir)
else:
    print('RUN_SMOKE_TEST=False -> skipped')


In [ ]:
# ============================================================
# CELL 8 — TRAINING MODE
# ============================================================

print('Một model duy nhất: random QP + mapped λ 1..64')
print('Full training được chạy ở Cell 11.')


In [ ]:
# 9) LEARNING CURVES
# Đồ thị Loss/BPP/Feature MSE được tạo sau khi Cell 11 hoàn thành.

print('Learning curves sẽ đọc history của model random-QP duy nhất.')


In [ ]:
# ============================================================
# CELL 10 — QP SWEEP
# Tắt trong phiên FULL TRAINING
# ============================================================

RUN_EVAL_SWEEP = False

if RUN_EVAL_SWEEP:
    print("QP evaluation enabled.")
else:
    print(
        "RUN_EVAL_SWEEP=False "
        "-> skip QP sweep during training."
    )

In [ ]:
# ============================================================
# CELL 10b — BASELINE EVALUATION
# Tắt trong phiên FULL TRAINING
# ============================================================

RUN_BASELINE_SWEEP = False

if RUN_BASELINE_SWEEP:
    print("Baseline evaluation enabled.")
else:
    print(
        "RUN_BASELINE_SWEEP=False "
        "-> skip baseline evaluation during training."
    )

In [ ]:
# ============================================================
# CELL 11 — FULL TRAINING MỘT MODEL VARIABLE-RATE
# Tối đa 40 epoch; early stopping theo val loss; random QP 0..63
# ============================================================

import json
import subprocess
import time
import torch
from pathlib import Path

OUTPUT_ROOT = Path(CFG['output_root'])
RUN_DIR = OUTPUT_ROOT / 'random_qp_lambda_1_64'
RUN_DIR.mkdir(parents=True, exist_ok=True)

last_ckpt = RUN_DIR / 'last.pth.tar'
best_ckpt = RUN_DIR / 'best.pth.tar'
history_file = RUN_DIR / 'history.jsonl'
resume_path = ''

if not RUN_FULL_TRAIN:
    print('RUN_FULL_TRAIN=False -> skipped')
else:
    if AUTO_RESUME and last_ckpt.exists():
        meta = torch.load(last_ckpt, map_location='cpu', weights_only=False)
        assert bool(meta.get('epoch_complete', False)), (
            'Checkpoint dừng giữa epoch hoặc thuộc phiên bản cũ; không được resume.'
        )
        saved_args = meta.get('args', {})
        for key in ('qp_min', 'qp_max', 'lambda_min', 'lambda_max', 'p_frames'):
            assert float(saved_args.get(key)) == float(CFG[key]), (
                f'Checkpoint dùng {key} khác experiment hiện tại.'
            )

        completed_epoch = int(meta.get('epoch', -1))
        if bool(meta.get('early_stopped', False)):
            print(f'Model đã hội tụ và early-stop sau epoch {completed_epoch + 1}.')
        elif completed_epoch + 1 >= CFG['epochs']:
            print(f'Đã train đủ trần {CFG["epochs"]} epoch.')
        else:
            resume_path = str(last_ckpt)
            print(f'Resume sau epoch {completed_epoch + 1}:', resume_path)

    if not last_ckpt.exists() or resume_path:
        start = time.time()
        subprocess.run(make_cmd(
            RUN_DIR,
            epochs=CFG['epochs'],
            max_steps=MAX_STEPS,
            max_train_samples=CFG['max_train_samples'],
            resume=resume_path,
        ), check=True)
        print(f'Training time: {(time.time() - start) / 3600:.2f} hours')

    rows = [
        json.loads(line)
        for line in history_file.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    completed = [row for row in rows if row.get('epoch_complete')]
    epoch_numbers = [
        int(row.get('epoch_number', row['epoch'] + 1))
        for row in completed
    ]

    assert epoch_numbers == list(range(1, len(completed) + 1)), epoch_numbers
    assert 1 <= len(completed) <= CFG['epochs']
    assert len(list((RUN_DIR / 'epoch_metrics').glob('epoch_*.json'))) == len(completed)
    assert len(list((RUN_DIR / 'epoch_plots').glob('epoch_*.png'))) == len(completed)
    assert last_ckpt.is_file() and best_ckpt.is_file()

    final_meta = torch.load(last_ckpt, map_location='cpu', weights_only=False)
    print('Training hoàn tất:', RUN_DIR)
    print('Early stopped:', bool(final_meta.get('early_stopped', False)))
    print('Completed epochs:', len(completed))
    print('Best checkpoint:', best_ckpt)


In [ ]:
# 12) TỔNG HỢP EPOCH + EXPORT CHECKPOINT
from pathlib import Path
import json
import shutil
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rows = [
    json.loads(line)
    for line in history_file.read_text(encoding='utf-8').splitlines()
    if line.strip() and json.loads(line).get('epoch_complete')
]
assert 1 <= len(rows) <= CFG['epochs']

epoch_rows = [{
    'epoch': row.get('epoch_number', row['epoch'] + 1),
    'global_step': row['global_step'],
    'train_loss': row['train']['loss'],
    'train_bpp': row['train']['bpp'],
    'train_feature_mse': row['train']['feature_mse'],
    'val_loss': row['val']['loss'],
    'val_bpp': row['val']['bpp'],
    'val_feature_mse': row['val']['feature_mse'],
} for row in rows]

history_df = pd.DataFrame(epoch_rows)
display(history_df)

history_csv = OUTPUT_ROOT / 'epoch_history.csv'
history_df.to_csv(history_csv, index=False)

checkpoint = torch.load(best_ckpt, map_location='cpu', weights_only=False)
video_only = RUN_DIR / 'dcvc_rt_vcm_random_qp_lambda_1_64_video_only.pth.tar'
torch.save({
    'state_dict': checkpoint['p_net'],
    'lambda_range': [CFG['lambda_min'], CFG['lambda_max']],
    'lambda_mapping': 'lambda(q)=64^(q/63)',
    'qp_sampling': ['uniform', CFG['qp_min'], CFG['qp_max']],
    'hierarchical_qp': True,
    'hierarchical_qp_offsets': QP_OFFSETS,
    'hierarchical_distortion_weights': DISTORTION_WEIGHTS,
}, video_only)

figure, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for axis, metric, title in zip(
    axes,
    ('loss', 'bpp', 'feature_mse'),
    ('Total Loss', 'BPP', 'Feature MSE'),
):
    axis.plot(history_df['epoch'], history_df[f'train_{metric}'], 'o-', label='Train')
    axis.plot(history_df['epoch'], history_df[f'val_{metric}'], 's--', label='Validation')
    axis.set(xlabel='Epoch', ylabel=title, title=title)
    axis.grid(alpha=0.3)
    axis.legend()

figure.suptitle('Random QP 0..63 — mapped λ 1..64')
figure.tight_layout()
curve_path = OUTPUT_ROOT / 'training_curves.png'
figure.savefig(curve_path, dpi=200, bbox_inches='tight')
plt.show()

zip_path = shutil.make_archive(
    '/kaggle/working/dcvc_rt_vcm_random_qp_lambda_1_64_7frame',
    'zip',
    root_dir=OUTPUT_ROOT,
)

print('History:', history_csv)
print('Video-only:', video_only)
print('Curves:', curve_path)
print('ZIP:', zip_path)


## Final evaluation: actual BPP + mAP

Notebook hiện tại có thể chạy ngay với **Vimeo-90K + Image checkpoint + Video checkpoint** và cho validation `estimated BPP / feature MSE / PSNR`.

Để báo cáo kết quả cuối cùng của nghiên cứu cần thêm **dataset object-detection có ground-truth**, vì Vimeo-90K không cung cấp bounding-box labels để tính mAP.

Khi bạn attach SFU-HW-Objects-v1 hoặc TVD (hoặc dataset detection khác), dùng checkpoint:

- Image: checkpoint DCVC-RT gốc (frozen).
- Video: `dcvc_rt_vcm_random_qp_lambda_1_64_video_only.pth.tar` vừa export.
- YOLO front-end: `student_front` nằm trong `best.pth.tar`.

Một checkpoint duy nhất được đánh giá ở nhiều QP. Final comparison phải dùng cùng test videos / detector / labels cho tất cả codec và vẽ **actual BPP vs mAP**. Cell dưới đây chỉ chuẩn bị rANS; **không cần chạy khi training**.

In [ ]:
# 13) OPTIONAL — build rANS entropy coder cho actual bitstream evaluation sau training
# Không chạy cell này nếu bạn mới chỉ training/validation.
BUILD_RANS = False

if BUILD_RANS:
    import subprocess, sys
    cpp_dir = DCVC_RT/'src'/'cpp'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '.'], cwd=str(cpp_dir), check=True)
    print('Built MLCodec_extensions_cpp.')
else:
    print('BUILD_RANS=False -> skipped')

### Resume / output quan trọng

Notebook lưu sau mỗi epoch hoàn chỉnh trong `random_qp_lambda_1_64/`:

- `epoch_metrics/epoch_XX.json`
- `epoch_plots/epoch_XX_loss_bpp_feature_mse.png`
- `history.jsonl` và `history.csv`
- `best.pth.tar`, `last.pth.tar`
- `dcvc_rt_vcm_random_qp_lambda_1_64_video_only.pth.tar`

Checkpoint cũ của bốn λ độc lập không tương thích với mapping mới. Giữ `batch_per_gpu=2`, `grad_accum=1` (global batch 4), `p_frames=6`, `crop_size=256` và `MAX_STEPS=0` cho experiment chính.

Early stopping dùng `val loss`, `patience=5`, `min_delta=1e-4`, với trần an toàn 40 epoch.
